## Product Recommendation Modeling & Lead Generation

This notebook builds product recommendations and lead scores for B2B customers.
The business idea is:
1. Use customer behavior, cohort, value, activity, and Salesforce features.
2. Train product-level models that estimate how likely a customer is to buy each product group.
3. Exclude products the customer has already purchased.
4. Rank the remaining products as recommendations.
5. Combine model probability with business value and customer activity into a lead score.

Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FEATURE_DIR = Path("../data/feature_engineering")
CLUSTER_DIR = Path("../data/cohort_clustering")
OUTPUT_DIR = Path("../data/product_recommendation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TOP_K = 3
MIN_POSITIVES = 25

print("Feature directory:", FEATURE_DIR.resolve())
print("Cluster directory:", CLUSTER_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


Feature directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\feature_engineering
Cluster directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\cohort_clustering
Output directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\product_recommendation


Load modeling inputs

In [2]:
df_customers = pd.read_csv(CLUSTER_DIR / "df_clustering_output.csv")
cust_prod_matrix_grp_1 = pd.read_csv(FEATURE_DIR / "cust_prod_matrix_grp_1.csv")
cust_prod_matrix_grp_3 = pd.read_csv(FEATURE_DIR / "cust_prod_matrix_grp_3.csv")

print("Customer feature table:", df_customers.shape)
print("Product group 1 matrix:", cust_prod_matrix_grp_1.shape)
print("Product group 3 matrix:", cust_prod_matrix_grp_3.shape)

display(df_customers.head())
display(cust_prod_matrix_grp_1.head())
display(cust_prod_matrix_grp_3.head())

Customer feature table: (1993, 85)
Product group 1 matrix: (1993, 7)
Product group 3 matrix: (1993, 36)


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized,active_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,is_inactive_6m,is_inactive_12m,avg_monthly_revenue,historical_ltv,purchase_freq_per_year,R_recency_m,F_frequency_m,M_sales,sf_active_months,sf_active_in_last_3m,sf_active_in_last_6m,sf_active_in_last_12m,sf_total_events,sf_total_selling_time,sf_total_activity_time,sf_activity_ratio_lifetime,sf_events_per_active_month,has_salesforce_activity,sales_without_recent_sf_activity,sf_activity_without_recent_sales,sales_value_segment,recency_segment,purchase_share_grp1_Automation Modules,purchase_share_grp1_Core Products,purchase_share_grp1_Maintenance Kits,purchase_share_grp1_Premium Products,purchase_share_grp1_Safety Solutions,purchase_share_grp1_Service Contracts,purchase_share_grp3_Calibration Kits Type 1,purchase_share_grp3_Calibration Kits Type 2,purchase_share_grp3_Compliance Safety Type 1,purchase_share_grp3_Compliance Safety Type 2,purchase_share_grp3_Controllers Type 1,purchase_share_grp3_Controllers Type 2,purchase_share_grp3_Core Advanced Type 1,purchase_share_grp3_Core Advanced Type 2,purchase_share_grp3_Core Basic Type 1,purchase_share_grp3_Core Basic Type 2,purchase_share_grp3_Core Replacement Type 1,purchase_share_grp3_Core Replacement Type 2,purchase_share_grp3_Extended Support Type 1,purchase_share_grp3_Extended Support Type 2,purchase_share_grp3_Facility Safety Type 2,purchase_share_grp3_Installation Type 1,purchase_share_grp3_Installation Type 2,purchase_share_grp3_Monitoring Type 1,purchase_share_grp3_Monitoring Type 2,purchase_share_grp3_Personal Safety Type 1,purchase_share_grp3_Personal Safety Type 2,purchase_share_grp3_Premium Custom Type 1,purchase_share_grp3_Premium Custom Type 2,purchase_share_grp3_Premium Plus Type 1,purchase_share_grp3_Premium Plus Type 2,purchase_share_grp3_Premium Standard Type 1,purchase_share_grp3_Premium Standard Type 2,purchase_share_grp3_Preventive Kits Type 1,purchase_share_grp3_Preventive Kits Type 2,purchase_share_grp3_Repair Kits Type 1,purchase_share_grp3_Repair Kits Type 2,purchase_share_grp3_Sensors Type 1,purchase_share_grp3_Sensors Type 2,purchase_share_grp3_Training Type 1,purchase_share_grp3_Training Type 2,cluster,cohort_ML
0,C00001,"144,433.2700",653,47,23,5,0.5714,0.6667,0.2778,0.1429,"3,073.0483",221.1842,23,34,1,0.6765,7.0000,3.0000,0.5000,3.0000,0,0,"4,248.0374","144,433.2700",8.1176,1,23,"144,433.2700",29,1,1,1,117,81.9200,67.5900,0.8056,4.0345,1,0,0,Top Value,Very Recent,0.1914,0.1654,0.1654,0.4778,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1654,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2665,0.0000,0.0000,0.2113,0.0000,0.0000,0.1654,0.0000,0.1914,0.0000,0.0000,0.0000,0,Emerging
1,C00002,"4,738.6100",21,17,8,9,1.0000,0.6667,0.3333,0.2286,278.7418,225.6481,8,31,1,0.2581,3.0000,1.0000,3.2857,14.0000,0,0,152.8584,"4,738.6100",3.0968,1,8,"4,738.6100",13,1,1,1,45,28.8700,18.8400,0.4333,3.4615,1,0,0,Low Value,Very Recent,0.3333,0.3333,0.0000,0.1905,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1905,0.0000,0.1429,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0,Emerging
2,C00003,"199,361.8000",658,105,31,11,0.5714,0.8333,0.5000,0.2857,"1,898.6838",302.9815,31,35,1,0.8857,15.0000,4.0000,0.1333,1.0000,0,0,"5,696.0514","199,361.8000",10.6286,1,31,"199,361.8000",24,1,1,1,124,76.3000,70.7800,0.7273,5.1667,1,0,0,Top Value,Very Recent,0.0000,0.3283,0.1201,0.1505,0.1474,0.2538,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0881,0.2401,0.0000,0.0000,0.0000,

,ID_Customer,purchase_flag_grp1_Automation Modules,purchase_flag_grp1_Core Products,purchase_flag_grp1_Maintenance Kits,purchase_flag_grp1_Premium Products,purchase_flag_grp1_Safety Solutions,purchase_flag_grp1_Service Contracts
0,C00001,1,1,1,1,0,0
1,C00002,1,1,0,1,1,0
2,C00003,0,1,1,1,1,1
3,C00004,1,1,1,0,0,0
4,C00005,1,1,0,1,0,1


,ID_Customer,purchase_flag_grp3_Calibration Kits Type 1,purchase_flag_grp3_Calibration Kits Type 2,purchase_flag_grp3_Compliance Safety Type 1,purchase_flag_grp3_Compliance Safety Type 2,purchase_flag_grp3_Controllers Type 1,purchase_flag_grp3_Controllers Type 2,purchase_flag_grp3_Core Advanced Type 1,purchase_flag_grp3_Core Advanced Type 2,purchase_flag_grp3_Core Basic Type 1,purchase_flag_grp3_Core Basic Type 2,purchase_flag_grp3_Core Replacement Type 1,purchase_flag_grp3_Core Replacement Type 2,purchase_flag_grp3_Extended Support Type 1,purchase_flag_grp3_Extended Support Type 2,purchase_flag_grp3_Facility Safety Type 2,purchase_flag_grp3_Installation Type 1,purchase_flag_grp3_Installation Type 2,purchase_flag_grp3_Monitoring Type 1,purchase_flag_grp3_Monitoring Type 2,purchase_flag_grp3_Personal Safety Type 1,purchase_flag_grp3_Personal Safety Type 2,purchase_flag_grp3_Premium Custom Type 1,purchase_flag_grp3_Premium Custom Type 2,purchase_flag_grp3_Premium Plus Type 1,purchase_flag_grp3_Premium Plus Type 2,purchase_flag_grp3_Premium Standard Type 1,purchase_flag_grp3_Premium Standard Type 2,purchase_flag_grp3_Preventive Kits Type 1,purchase_flag_grp3_Preventive Kits Type 2,purchase_flag_grp3_Repair Kits Type 1,purchase_flag_grp3_Repair Kits Type 2,purchase_flag_grp3_Sensors Type 1,purchase_flag_grp3_Sensors Type 2,purchase_flag_grp3_Training Type 1,purchase_flag_grp3_Training Type 2
0,C00001,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0
1,C00002,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0
2,C00003,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,1,0
3,C00004,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,C00005,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0


Inspect Recommendation Targets

In [ ]:
# understand imbalances
def target_summary(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    target_cols = [c for c in matrix.columns if c.startswith(prefix)]
    rows = []
    for col in target_cols:
        y = matrix[col].astype(int)
        rows.append({
            "target_col": col,
            "product": col.replace(prefix, ""),
            "customers": len(y),
            "positive_customers": int(y.sum()),
            "positive_rate": float(y.mean()),
            "negative_customers": int((1 - y).sum())})
    return pd.DataFrame(rows).sort_values("positive_rate", ascending=False)


grp1_target_summary = target_summary(cust_prod_matrix_grp_1, "purchase_flag_grp1_")
grp3_target_summary = target_summary(cust_prod_matrix_grp_3, "purchase_flag_grp3_")

display(grp1_target_summary)
display(grp3_target_summary.head(20))

,target_col,product,customers,positive_customers,positive_rate,negative_customers
0,purchase_flag_grp1_Automation Modules,Automation Modules,1993,1656,0.8309,337
1,purchase_flag_grp1_Core Products,Core Products,1993,1622,0.8138,371
2,purchase_flag_grp1_Maintenance Kits,Maintenance Kits,1993,1310,0.6573,683
5,purchase_flag_grp1_Service Contracts,Service Contracts,1993,1174,0.5891,819
3,purchase_flag_grp1_Premium Products,Premium Products,1993,1055,0.5294,938
4,purchase_flag_grp1_Safety Solutions,Safety Solutions,1993,909,0.4561,1084


,target_col,product,customers,positive_customers,positive_rate,negative_customers
7,purchase_flag_grp3_Core Advanced Type 2,Core Advanced Type 2,1993,695,0.3487,1298
4,purchase_flag_grp3_Controllers Type 1,Controllers Type 1,1993,684,0.3432,1309
9,purchase_flag_grp3_Core Basic Type 2,Core Basic Type 2,1993,678,0.3402,1315
31,purchase_flag_grp3_Sensors Type 1,Sensors Type 1,1993,669,0.3357,1324
6,purchase_flag_grp3_Core Advanced Type 1,Core Advanced Type 1,1993,609,0.3056,1384
8,purchase_flag_grp3_Core Basic Type 1,Core Basic Type 1,1993,601,0.3016,1392
32,purchase_flag_grp3_Sensors Type 2,Sensors Type 2,1993,586,0.2940,1407
5,purchase_flag_grp3_Controllers Type 2,Controllers Type 2,1993,585,0.2935,1408
17,purchase_flag_grp3_Monitoring Type 1,Monitoring Type 1,1993,459,0.2303,1534
30,purchase_flag_grp3_Repair Kits Type 2,Repair Kits Type 2,1993,428,0.2148,1565


Build Modeling Tables

In [ ]:
# merge customer features with product target matrix

def build_modeling_table(customer_df: pd.DataFrame, target_matrix: pd.DataFrame) -> pd.DataFrame:
    # validate one row per customer before merging
    if customer_df["ID_Customer"].duplicated().any():
        raise ValueError("customer_df contains duplicated ID_Customer values.")
    if target_matrix["ID_Customer"].duplicated().any():
        raise ValueError("target_matrix contains duplicated ID_Customer values.")

    return customer_df.merge(target_matrix, on="ID_Customer", how="inner", validate="one_to_one")


df_model_grp1 = build_modeling_table(df_customers, cust_prod_matrix_grp_1)
df_model_grp3 = build_modeling_table(df_customers, cust_prod_matrix_grp_3)

print("Modeling table group 1:", df_model_grp1.shape)
print("Modeling table group 3:", df_model_grp3.shape)

display(df_model_grp1.head())

Modeling table group 1: (1993, 91)
Modeling table group 3: (1993, 120)


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized,active_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,is_inactive_6m,is_inactive_12m,avg_monthly_revenue,historical_ltv,purchase_freq_per_year,R_recency_m,F_frequency_m,M_sales,sf_active_months,sf_active_in_last_3m,sf_active_in_last_6m,sf_active_in_last_12m,sf_total_events,sf_total_selling_time,sf_total_activity_time,sf_activity_ratio_lifetime,sf_events_per_active_month,has_salesforce_activity,sales_without_recent_sf_activity,sf_activity_without_recent_sales,sales_value_segment,recency_segment,purchase_share_grp1_Automation Modules,purchase_share_grp1_Core Products,purchase_share_grp1_Maintenance Kits,purchase_share_grp1_Premium Products,purchase_share_grp1_Safety Solutions,purchase_share_grp1_Service Contracts,purchase_share_grp3_Calibration Kits Type 1,purchase_share_grp3_Calibration Kits Type 2,purchase_share_grp3_Compliance Safety Type 1,purchase_share_grp3_Compliance Safety Type 2,purchase_share_grp3_Controllers Type 1,purchase_share_grp3_Controllers Type 2,purchase_share_grp3_Core Advanced Type 1,purchase_share_grp3_Core Advanced Type 2,purchase_share_grp3_Core Basic Type 1,purchase_share_grp3_Core Basic Type 2,purchase_share_grp3_Core Replacement Type 1,purchase_share_grp3_Core Replacement Type 2,purchase_share_grp3_Extended Support Type 1,purchase_share_grp3_Extended Support Type 2,purchase_share_grp3_Facility Safety Type 2,purchase_share_grp3_Installation Type 1,purchase_share_grp3_Installation Type 2,purchase_share_grp3_Monitoring Type 1,purchase_share_grp3_Monitoring Type 2,purchase_share_grp3_Personal Safety Type 1,purchase_share_grp3_Personal Safety Type 2,purchase_share_grp3_Premium Custom Type 1,purchase_share_grp3_Premium Custom Type 2,purchase_share_grp3_Premium Plus Type 1,purchase_share_grp3_Premium Plus Type 2,purchase_share_grp3_Premium Standard Type 1,purchase_share_grp3_Premium Standard Type 2,purchase_share_grp3_Preventive Kits Type 1,purchase_share_grp3_Preventive Kits Type 2,purchase_share_grp3_Repair Kits Type 1,purchase_share_grp3_Repair Kits Type 2,purchase_share_grp3_Sensors Type 1,purchase_share_grp3_Sensors Type 2,purchase_share_grp3_Training Type 1,purchase_share_grp3_Training Type 2,cluster,cohort_ML,purchase_flag_grp1_Automation Modules,purchase_flag_grp1_Core Products,purchase_flag_grp1_Maintenance Kits,purchase_flag_grp1_Premium Products,purchase_flag_grp1_Safety Solutions,purchase_flag_grp1_Service Contracts
0,C00001,"144,433.2700",653,47,23,5,0.5714,0.6667,0.2778,0.1429,"3,073.0483",221.1842,23,34,1,0.6765,7.0000,3.0000,0.5000,3.0000,0,0,"4,248.0374","144,433.2700",8.1176,1,23,"144,433.2700",29,1,1,1,117,81.9200,67.5900,0.8056,4.0345,1,0,0,Top Value,Very Recent,0.1914,0.1654,0.1654,0.4778,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1654,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2665,0.0000,0.0000,0.2113,0.0000,0.0000,0.1654,0.0000,0.1914,0.0000,0.0000,0.0000,0,Emerging,1,1,1,1,0,0
1,C00002,"4,738.6100",21,17,8,9,1.0000,0.6667,0.3333,0.2286,278.7418,225.6481,8,31,1,0.2581,3.0000,1.0000,3.2857,14.0000,0,0,152.8584,"4,738.6100",3.0968,1,8,"4,738.6100",13,1,1,1,45,28.8700,18.8400,0.4333,3.4615,1,0,0,Low Value,Very Recent,0.3333,0.3333,0.0000,0.1905,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1905,0.0000,0.1429,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0,Emerging,1,1,0,1,1,0
2,C00003,"199,361.8000",658,105,31,11,0.5714,0.8333,0.5000,0.2857,"1,898.6838",302.9815,31,35,1,0.8857,15.0000,4.0000,0.1333,1.0000,0,0,"5,696.0514",

Define Feature Exclusions

In [6]:
def get_target_cols(df: pd.DataFrame, target_prefix: str) -> list[str]:
    return [c for c in df.columns if c.startswith(target_prefix)]

def base_columns_to_drop(df: pd.DataFrame) -> list[str]:
    drop_cols = []

    # remove ID, date, period columns
    drop_cols += [c for c in df.columns if c.lower() in ["id_customer", "id_product"]]
    drop_cols += [c for c in df.columns if "date" in c.lower()]
    drop_cols += [c for c in df.columns if c.lower().endswith("_ym")]
    drop_cols += [c for c in df.columns if c.lower().endswith("_ym_ord")]

    # remove duplicate column names while preserving order
    return list(dict.fromkeys([c for c in drop_cols if c in df.columns]))


grp1_target_cols = get_target_cols(df_model_grp1, "purchase_flag_grp1_")
grp3_target_cols = get_target_cols(df_model_grp3, "purchase_flag_grp3_")

print("Group 1 targets:", len(grp1_target_cols))
print("Group 3 targets:", len(grp3_target_cols))
print("Base drop columns example:", base_columns_to_drop(df_model_grp1)[:20])


Group 1 targets: 6
Group 3 targets: 35
Base drop columns example: ['ID_Customer']


Preprocessing Helper 

In [8]:
def make_preprocessor(X: pd.DataFrame) -> tuple[ColumnTransformer, list[str], list[str]]:
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())])

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols)],
        remainder="drop")

    return preprocessor, numeric_cols, categorical_cols

Baseline Model: Popularity Ranking

In [ ]:
def popularity_baseline_score(y_valid: pd.Series, train_positive_rate: float) -> dict:
    # every customer receives the same probability: the product's training positive rate
    baseline_proba = np.repeat(train_positive_rate, len(y_valid))

    try:
        baseline_ap = average_precision_score(y_valid, baseline_proba) 
    except ValueError:
        baseline_ap = np.nan

    try:
        baseline_auc = roc_auc_score(y_valid, baseline_proba)
    except ValueError:
        baseline_auc = np.nan

    return {
        "baseline_avg_precision": baseline_ap,
        "baseline_roc_auc": baseline_auc}

Train and Evaluate One Product Model

In [ ]:
def train_one_product_model(
    df: pd.DataFrame,
    target_col: str,
    all_target_cols: list[str],
    product_prefix: str,
) -> tuple[Pipeline | None, dict]:
    y = df[target_col].astype(int)
    product_name = target_col.replace(product_prefix, "")

    # Skip products with too little signal
    if y.nunique() < 2 or y.sum() < MIN_POSITIVES:
        return None, {
            "product": product_name,
            "target_col": target_col,
            "status": "skipped_low_support",
            "n_total": len(y),
            "positive_customers": int(y.sum()),
            "positive_rate": float(y.mean()),
        }

    leakage_cols = all_target_cols
    drop_cols = base_columns_to_drop(df) + leakage_cols
    X = df.drop(columns=drop_cols, errors="ignore").copy()

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y)

    baseline = popularity_baseline_score(y_valid, y_train.mean())
    preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_train)

    candidates = {
        "LogisticRegression": LogisticRegression(
            max_iter=2_000,
            class_weight="balanced",
            random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=350,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1)}

    fitted_models = []

    for model_name, model in candidates.items():
        pipe = Pipeline(
            steps=[
                ("preprocess", preprocessor),
                ("model", model)])

        pipe.fit(X_train, y_train)
        proba_valid = pipe.predict_proba(X_valid)[:, 1]
        pred_valid = (proba_valid >= 0.50).astype(int)

        try:
            avg_precision = average_precision_score(y_valid, proba_valid)
        except ValueError:
            avg_precision = np.nan

        try:
            roc_auc = roc_auc_score(y_valid, proba_valid)
        except ValueError:
            roc_auc = np.nan

        metrics = {
            "product": product_name,
            "target_col": target_col,
            "status": "trained",
            "model": model_name,
            "n_total": len(y),
            "positive_customers": int(y.sum()),
            "positive_rate": float(y.mean()),
            "train_rows": len(X_train),
            "valid_rows": len(X_valid),
            "valid_positive_customers": int(y_valid.sum()),
            "valid_avg_precision": avg_precision,
            "valid_roc_auc": roc_auc,
            "valid_precision_at_050": precision_score(y_valid, pred_valid, zero_division=0),
            "valid_recall_at_050": recall_score(y_valid, pred_valid, zero_division=0),
            "n_numeric_features": len(numeric_cols),
            "n_categorical_features": len(categorical_cols),
            **baseline}

        fitted_models.append((metrics, pipe))

    # select best candidate by average precision; use ROC-AUC as tie breaker
    # in lead generation, we care about ranking good leads near the top -
    # Average precision rewards models that assign high probabilities to true buyers
    best_metrics, best_pipe = max(
        fitted_models,
        key=lambda item: (
            np.nan_to_num(item[0]["valid_avg_precision"], nan=-1),
            np.nan_to_num(item[0]["valid_roc_auc"], nan=-1)))

    # refit the best model on all rows before scoring all customers.
    best_X = df.drop(columns=drop_cols, errors="ignore").copy()
    best_pipe.fit(best_X, y)

    return best_pipe, best_metrics

Generate Recommendations for One productg group

In [ ]:
""" For each target:
- train the best model
- score all customers
- keep only customers who have not bought the product yet
- create a long recommendation table """

def build_recommendations_for_group(
    df: pd.DataFrame,
    target_prefix: str,
    product_group_label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_cols = get_target_cols(df, target_prefix)
    all_recommendations = []
    model_registry_rows = []

    for target_col in target_cols:
        model, metrics = train_one_product_model(
            df=df,
            target_col=target_col,
            all_target_cols=target_cols,
            product_prefix=target_prefix)
        metrics["product_group"] = product_group_label
        model_registry_rows.append(metrics)

        if model is None:
            continue

        product_name = target_col.replace(target_prefix, "")
        drop_cols = base_columns_to_drop(df) + target_cols
        X_all = df.drop(columns=drop_cols, errors="ignore").copy()

        # rredict probability for every customer
        proba_all = model.predict_proba(X_all)[:, 1]

        # recommend only to customers who have not bought that product group yet
        not_bought_mask = df[target_col].astype(int).eq(0)

        recs = pd.DataFrame({
            "ID_Customer": df.loc[not_bought_mask, "ID_Customer"].values,
            "product_group": product_group_label,
            "recommended_product": product_name,
            "model_probability": proba_all[not_bought_mask],
            "source_target_col": target_col})

        all_recommendations.append(recs)

    if all_recommendations:
        recommendations_long = pd.concat(all_recommendations, ignore_index=True)
    else:
        recommendations_long = pd.DataFrame(
            columns=[
                "ID_Customer",
                "product_group",
                "recommended_product",
                "model_probability",
                "source_target_col"])

    model_registry = pd.DataFrame(model_registry_rows)
    return recommendations_long, model_registry

Train Product Group 1 Recommendation Models

In [ ]:
""" Product group 1 is a higher-level recommendation layer.
This is useful when sales teams need broader guidance, such as:
- recommend a product category
- identify cross-sell potential
- create simple sales plays """

recs_grp1_long, registry_grp1 = build_recommendations_for_group(
    df=df_model_grp1,
    target_prefix="purchase_flag_grp1_",
    product_group_label="grp1")

print("Group 1 recommendations:", recs_grp1_long.shape)
print("Group 1 model registry:", registry_grp1.shape)

display(registry_grp1.sort_values("valid_avg_precision", ascending=False))
display(recs_grp1_long.head())


Group 1 recommendations: (4232, 5)
Group 1 model registry: (6, 19)


,product,target_col,status,model,n_total,positive_customers,positive_rate,train_rows,valid_rows,valid_positive_customers,valid_avg_precision,valid_roc_auc,valid_precision_at_050,valid_recall_at_050,n_numeric_features,n_categorical_features,baseline_avg_precision,baseline_roc_auc,product_group
3,Premium Products,purchase_flag_grp1_Premium Products,trained,RandomForest,1993,1055,0.5294,1594,399,211,1.0000,1.0000,1.0000,1.0000,81,3,0.5288,0.5000,grp1
0,Automation Modules,purchase_flag_grp1_Automation Modules,trained,RandomForest,1993,1656,0.8309,1594,399,332,1.0000,1.0000,1.0000,1.0000,81,3,0.8321,0.5000,grp1
1,Core Products,purchase_flag_grp1_Core Products,trained,RandomForest,1993,1622,0.8138,1594,399,325,1.0000,1.0000,1.0000,1.0000,81,3,0.8145,0.5000,grp1
2,Maintenance Kits,purchase_flag_grp1_Maintenance Kits,trained,RandomForest,1993,1310,0.6573,1594,399,262,1.0000,1.0000,1.0000,1.0000,81,3,0.6566,0.5000,grp1
5,Service Contracts,purchase_flag_grp1_Service Contracts,trained,RandomForest,1993,1174,0.5891,1594,399,235,1.0000,1.0000,1.0000,1.0000,81,3,0.5890,0.5000,grp1
4,Safety Solutions,purchase_flag_grp1_Safety Solutions,trained,RandomForest,1993,909,0.4561,1594,399,182,1.0000,1.0000,1.0000,1.0000,81,3,0.4561,0.5000,grp1


,ID_Customer,product_group,recommended_product,model_probability,source_target_col
0,C00003,grp1,Automation Modules,0.1262,purchase_flag_grp1_Automation Modules
1,C00014,grp1,Automation Modules,0.0526,purchase_flag_grp1_Automation Modules
2,C00021,grp1,Automation Modules,0.0399,purchase_flag_grp1_Automation Modules
3,C00026,grp1,Automation Modules,0.0796,purchase_flag_grp1_Automation Modules
4,C00029,grp1,Automation Modules,0.1194,purchase_flag_grp1_Automation Modules


Train Product Group 3 Recommendation Models

In [15]:
""" Product group 3 is more granular.
This is more actionable but also harder:
- more targets
- lower positive rates per product
- stronger class imbalance
For that reason, some rare targets may be skipped if they do not have enough positive customers """

recs_grp3_long, registry_grp3 = build_recommendations_for_group(
    df=df_model_grp3,
    target_prefix="purchase_flag_grp3_",
    product_group_label="grp3")

print("Group 3 recommendations:", recs_grp3_long.shape)
print("Group 3 model registry:", registry_grp3.shape)

display(registry_grp3.sort_values("valid_avg_precision", ascending=False).head(20))
display(recs_grp3_long.head())

Group 3 recommendations: (56758, 5)
Group 3 model registry: (35, 19)


,product,target_col,status,model,n_total,positive_customers,positive_rate,train_rows,valid_rows,valid_positive_customers,valid_avg_precision,valid_roc_auc,valid_precision_at_050,valid_recall_at_050,n_numeric_features,n_categorical_features,baseline_avg_precision,baseline_roc_auc,product_group
0,Calibration Kits Type 1,purchase_flag_grp3_Calibration Kits Type 1,trained,RandomForest,1993,282,0.1415,1594,399,56,1.0000,1.0000,1.0000,1.0000,81,3,0.1404,0.5000,grp3
23,Premium Plus Type 1,purchase_flag_grp3_Premium Plus Type 1,trained,LogisticRegression,1993,239,0.1199,1594,399,48,1.0000,1.0000,1.0000,0.9792,81,3,0.1203,0.5000,grp3
17,Monitoring Type 1,purchase_flag_grp3_Monitoring Type 1,trained,RandomForest,1993,459,0.2303,1594,399,92,1.0000,1.0000,1.0000,0.9891,81,3,0.2306,0.5000,grp3
31,Sensors Type 1,purchase_flag_grp3_Sensors Type 1,trained,RandomForest,1993,669,0.3357,1594,399,134,1.0000,1.0000,1.0000,1.0000,81,3,0.3358,0.5000,grp3
28,Preventive Kits Type 2,purchase_flag_grp3_Preventive Kits Type 2,trained,LogisticRegression,1993,276,0.1385,1594,399,55,1.0000,1.0000,1.0000,0.9636,81,3,0.1378,0.5000,grp3
22,Premium Custom Type 2,purchase_flag_grp3_Premium Custom Type 2,trained,RandomForest,1993,387,0.1942,1594,399,77,1.0000,1.0000,1.0000,1.0000,81,3,0.1930,0.5000,grp3
3,Compliance Safety Type 2,purchase_flag_grp3_Compliance Safety Type 2,trained,LogisticRegression,1993,161,0.0808,1594,399,32,1.0000,1.0000,1.0000,1.0000,81,3,0.0802,0.5000,grp3
14,Facility Safety Type 2,purchase_flag_grp3_Facility Safety Type 2,trained,RandomForest,1993,263,0.1320,1594,399,53,1.0000,1.0000,1.0000,0.9623,81,3,0.1328,0.5000,grp3
16,Installation Type 2,purchase_flag_grp3_Installation Type 2,trained,RandomForest,1993,196,0.0983,1594,399,39,1.0000,1.0000,1.0000,0.8718,81,3,0.0977,0.5000,grp3
10,Core Replacement Type 1,purchase_flag_grp3_Core Replacement Type 1,trained,LogisticRegression,1993,346,0.1736,1594,399,69,1.0000,1.0000,1.0000,1.0000,81,3,0.1729,0.5000,grp3


,ID_Customer,product_group,recommended_product,model_probability,source_target_col
0,C00001,grp3,Calibration Kits Type 1,0.1150,purchase_flag_grp3_Calibration Kits Type 1
1,C00002,grp3,Calibration Kits Type 1,0.0492,purchase_flag_grp3_Calibration Kits Type 1
2,C00003,grp3,Calibration Kits Type 1,0.1558,purchase_flag_grp3_Calibration Kits Type 1
3,C00004,grp3,Calibration Kits Type 1,0.0956,purchase_flag_grp3_Calibration Kits Type 1
4,C00005,grp3,Calibration Kits Type 1,0.0198,purchase_flag_grp3_Calibration Kits Type 1


Review Model Performance

In [16]:
""" model registry helps us understand which product targets are reliable.
Useful columns:
- positive_rate: how common the product is
- valid_avg_precision: ranking quality
- baseline_avg_precision: popularity baseline
- valid_roc_auc: separation between buyers and non-buyers
- status: whether the model was trained or skipped """

model_registry_all = pd.concat([registry_grp1, registry_grp3], ignore_index=True)

model_registry_all["avg_precision_lift_vs_baseline"] = (
    model_registry_all["valid_avg_precision"] / model_registry_all["baseline_avg_precision"]
).replace([np.inf, -np.inf], np.nan)

display(
    model_registry_all
    .sort_values(["status", "valid_avg_precision"], ascending=[True, False])
    .head(30))

display(
    model_registry_all
    .groupby(["product_group", "status"], as_index=False)
    .agg(
        targets=("target_col", "count"),
        avg_positive_rate=("positive_rate", "mean"),
        avg_valid_avg_precision=("valid_avg_precision", "mean"),
        avg_baseline_avg_precision=("baseline_avg_precision", "mean"),
        avg_precision_lift=("avg_precision_lift_vs_baseline", "mean")))

,product,target_col,status,model,n_total,positive_customers,positive_rate,train_rows,valid_rows,valid_positive_customers,valid_avg_precision,valid_roc_auc,valid_precision_at_050,valid_recall_at_050,n_numeric_features,n_categorical_features,baseline_avg_precision,baseline_roc_auc,product_group,avg_precision_lift_vs_baseline
3,Premium Products,purchase_flag_grp1_Premium Products,trained,RandomForest,1993,1055,0.5294,1594,399,211,1.0000,1.0000,1.0000,1.0000,81,3,0.5288,0.5000,grp1,1.8910
6,Calibration Kits Type 1,purchase_flag_grp3_Calibration Kits Type 1,trained,RandomForest,1993,282,0.1415,1594,399,56,1.0000,1.0000,1.0000,1.0000,81,3,0.1404,0.5000,grp3,7.1250
23,Monitoring Type 1,purchase_flag_grp3_Monitoring Type 1,trained,RandomForest,1993,459,0.2303,1594,399,92,1.0000,1.0000,1.0000,0.9891,81,3,0.2306,0.5000,grp3,4.3370
29,Premium Plus Type 1,purchase_flag_grp3_Premium Plus Type 1,trained,LogisticRegression,1993,239,0.1199,1594,399,48,1.0000,1.0000,1.0000,0.9792,81,3,0.1203,0.5000,grp3,8.3125
37,Sensors Type 1,purchase_flag_grp3_Sensors Type 1,trained,RandomForest,1993,669,0.3357,1594,399,134,1.0000,1.0000,1.0000,1.0000,81,3,0.3358,0.5000,grp3,2.9776
0,Automation Modules,purchase_flag_grp1_Automation Modules,trained,RandomForest,1993,1656,0.8309,1594,399,332,1.0000,1.0000,1.0000,1.0000,81,3,0.8321,0.5000,grp1,1.2018
1,Core Products,purchase_flag_grp1_Core Products,trained,RandomForest,1993,1622,0.8138,1594,399,325,1.0000,1.0000,1.0000,1.0000,81,3,0.8145,0.5000,grp1,1.2277
2,Maintenance Kits,purchase_flag_grp1_Maintenance Kits,trained,RandomForest,1993,1310,0.6573,1594,399,262,1.0000,1.0000,1.0000,1.0000,81,3,0.6566,0.5000,grp1,1.5229
5,Service Contracts,purchase_flag_grp1_Service Contracts,trained,RandomForest,1993,1174,0.5891,1594,399,235,1.0000,1.0000,1.0000,1.0000,81,3,0.5890,0.5000,grp1,1.6979
9,Compliance Safety Type 2,purchase_flag_grp3_Compliance Safety Type 2,trained,LogisticRegression,1993,161,0.0808,1594,399,32,1.0000,1.0000,1.0000,1.0000,81,3,0.0802,0.5000,grp3,12.4688


,product_group,status,targets,avg_positive_rate,avg_valid_avg_precision,avg_baseline_avg_precision,avg_precision_lift
0,grp1,trained,6,0.6461,1.0000,0.6462,1.6223
1,grp3,trained,35,0.1863,0.9997,0.1861,7.1381


Rank Recommendations

In [ ]:
# combine grp 1 and grp 3 recommendations
# ranking is based first on model probability. later we combine it with business signals to create a lead score

recommendations_long = pd.concat([recs_grp1_long, recs_grp3_long], ignore_index=True)

recommendations_long = recommendations_long.sort_values(
    ["ID_Customer", "model_probability"],
    ascending=[True, False])

recommendations_long["model_rank_per_customer"] = (
    recommendations_long
    .groupby("ID_Customer")
    .cumcount()
    + 1)

recommendations_topk_model = recommendations_long[
    recommendations_long["model_rank_per_customer"] <= TOP_K
].copy()

print("All recommendation rows:", recommendations_long.shape)
print("Top-K model recommendation rows:", recommendations_topk_model.shape)

display(recommendations_topk_model.head(20))

All recommendation rows: (60990, 6)
Top-K model recommendation rows: (5979, 6)


,ID_Customer,product_group,recommended_product,model_probability,source_target_col,model_rank_per_customer
39872,C00001,grp3,Premium Custom Type 2,0.1752,purchase_flag_grp3_Premium Custom Type 2,1
43232,C00001,grp3,Premium Plus Type 2,0.1641,purchase_flag_grp3_Premium Plus Type 2,2
53397,C00001,grp3,Repair Kits Type 2,0.1550,purchase_flag_grp3_Repair Kits Type 2,3
7582,C00002,grp3,Compliance Safety Type 1,0.1792,purchase_flag_grp3_Compliance Safety Type 1,1
13810,C00002,grp3,Core Advanced Type 1,0.1677,purchase_flag_grp3_Core Advanced Type 1,2
26217,C00002,grp3,Facility Safety Type 2,0.1640,purchase_flag_grp3_Facility Safety Type 2,3
17883,C00003,grp3,Core Basic Type 2,0.2303,purchase_flag_grp3_Core Basic Type 2,1
34536,C00003,grp3,Personal Safety Type 1,0.2126,purchase_flag_grp3_Personal Safety Type 1,2
39874,C00003,grp3,Premium Custom Type 2,0.1886,purchase_flag_grp3_Premium Custom Type 2,3
17884,C00004,grp3,Core Basic Type 2,0.1898,purchase_flag_grp3_Core Basic Type 2,1


CReate wide Recommendation Tables

In [18]:
# for dashboard and final customer output

def make_topk_wide(recs_long: pd.DataFrame, product_group: str, top_k: int = TOP_K) -> pd.DataFrame:
    part = recs_long[recs_long["product_group"] == product_group].copy()

    if part.empty:
        return pd.DataFrame(columns=["ID_Customer"])

    part = part.sort_values(
        ["ID_Customer", "model_probability"],
        ascending=[True, False])
    part["rank"] = part.groupby("ID_Customer").cumcount() + 1
    part = part[part["rank"] <= top_k].copy()

    wide_parts = []
    for k in range(1, top_k + 1):
        kth = (
            part[part["rank"] == k]
            [["ID_Customer", "recommended_product", "model_probability", "rank"]]
            .rename(columns={
                "recommended_product": f"reco_top{k}_{product_group}_product",
                "model_probability": f"reco_top{k}_{product_group}_probability",
                "rank": f"reco_top{k}_{product_group}_rank"}))
        wide_parts.append(kth)

    wide = wide_parts[0]
    for kth in wide_parts[1:]:
        wide = wide.merge(kth, on="ID_Customer", how="outer")

    return wide


recos_grp1_top3_wide = make_topk_wide(recommendations_long, "grp1", TOP_K)
recos_grp3_top3_wide = make_topk_wide(recommendations_long, "grp3", TOP_K)

display(recos_grp1_top3_wide.head())
display(recos_grp3_top3_wide.head())

,ID_Customer,reco_top1_grp1_product,reco_top1_grp1_probability,reco_top1_grp1_rank,reco_top2_grp1_product,reco_top2_grp1_probability,reco_top2_grp1_rank,reco_top3_grp1_product,reco_top3_grp1_probability,reco_top3_grp1_rank
0,C00001,Service Contracts,0.0747,1,Safety Solutions,0.0644,2.0000,NaN,NaN,NaN
1,C00002,Maintenance Kits,0.0861,1,Service Contracts,0.0836,2.0000,NaN,NaN,NaN
2,C00003,Automation Modules,0.1262,1,NaN,NaN,NaN,NaN,NaN,NaN
3,C00004,Premium Products,0.0707,1,Safety Solutions,0.0702,2.0000,Service Contracts,0.0353,3.0000
4,C00005,Safety Solutions,0.0795,1,Maintenance Kits,0.0511,2.0000,NaN,NaN,NaN


,ID_Customer,reco_top1_grp3_product,reco_top1_grp3_probability,reco_top1_grp3_rank,reco_top2_grp3_product,reco_top2_grp3_probability,reco_top2_grp3_rank,reco_top3_grp3_product,reco_top3_grp3_probability,reco_top3_grp3_rank
0,C00001,Premium Custom Type 2,0.1752,1,Premium Plus Type 2,0.1641,2,Repair Kits Type 2,0.1550,3
1,C00002,Compliance Safety Type 1,0.1792,1,Core Advanced Type 1,0.1677,2,Facility Safety Type 2,0.1640,3
2,C00003,Core Basic Type 2,0.2303,1,Personal Safety Type 1,0.2126,2,Premium Custom Type 2,0.1886,3
3,C00004,Core Basic Type 2,0.1898,1,Controllers Type 1,0.1674,2,Monitoring Type 1,0.1459,3
4,C00005,Training Type 1,0.1787,1,Extended Support Type 1,0.1652,2,Training Type 2,0.1428,3


Build Lead Scoring Dataset

In [19]:
""" A product recommendation probability alone is not enough.
Sales teams also care about: customer value, recent activity, churn or inactivity risk, cohort context.
The lead score combines model probability with business value and customer engagement. """

customer_context_cols = [
    "ID_Customer",
    "cohort_ML",
    "cluster",
    "Sales",
    "historical_ltv",
    "avg_monthly_revenue",
    "active_months",
    "months_since_last_purchase",
    "activity_ratio_lifetime",
    "avg_inactive_gap_months",
    "prd_grp_1_coverage_pct",
    "prd_grp_3_coverage_pct",
    "sf_total_events",
    "sf_active_months",
    "sf_months_since_last_event",
    "sf_activity_without_recent_sales",
    "sales_without_recent_sf_activity"]

customer_context_cols = [c for c in customer_context_cols if c in df_customers.columns]

leads_long = recommendations_long.merge(
    df_customers[customer_context_cols],
    on="ID_Customer",
    how="left",
    validate="many_to_one")

display(leads_long.head())

,ID_Customer,product_group,recommended_product,model_probability,source_target_col,model_rank_per_customer,cohort_ML,cluster,Sales,historical_ltv,avg_monthly_revenue,active_months,months_since_last_purchase,activity_ratio_lifetime,avg_inactive_gap_months,prd_grp_1_coverage_pct,prd_grp_3_coverage_pct,sf_total_events,sf_active_months,sf_activity_without_recent_sales,sales_without_recent_sf_activity
0,C00001,grp3,Premium Custom Type 2,0.1752,purchase_flag_grp3_Premium Custom Type 2,1,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
1,C00001,grp3,Premium Plus Type 2,0.1641,purchase_flag_grp3_Premium Plus Type 2,2,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
2,C00001,grp3,Repair Kits Type 2,0.1550,purchase_flag_grp3_Repair Kits Type 2,3,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
3,C00001,grp3,Sensors Type 2,0.1530,purchase_flag_grp3_Sensors Type 2,4,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
4,C00001,grp3,Monitoring Type 1,0.1192,purchase_flag_grp3_Monitoring Type 1,5,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0


**Lead Score Formula**

This project uses a transparent scoring formula.
Lead Score components:
- model_score: model probability for the recommendation
- value_score: normalized customer value
- activity_score: recent and frequent purchasing behavior
- sf_engagement_score: Salesforce engagement
- risk_penalty: inactivity penalty

The weights are business assumptions. In a real company, they should be aligned with sales capacity and validated against conversion outcomes.

In [20]:
def minmax_score(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").astype(float)
    min_value = s.min(skipna=True)
    max_value = s.max(skipna=True)

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.0, index=s.index)

    return ((s - min_value) / (max_value - min_value)).fillna(0)


leads_long["model_score"] = leads_long["model_probability"].clip(0, 1)

if "historical_ltv" in leads_long.columns:
    leads_long["value_score"] = minmax_score(leads_long["historical_ltv"])
else:
    leads_long["value_score"] = minmax_score(leads_long["Sales"])

# Recency: lower months_since_last_purchase is better.
if "months_since_last_purchase" in leads_long.columns:
    leads_long["recency_score"] = 1 - minmax_score(leads_long["months_since_last_purchase"])
else:
    leads_long["recency_score"] = 0.0

if "active_months" in leads_long.columns:
    leads_long["frequency_score"] = minmax_score(leads_long["active_months"])
else:
    leads_long["frequency_score"] = 0.0

leads_long["activity_score"] = (
    0.60 * leads_long["recency_score"]
    + 0.40 * leads_long["frequency_score"])

if "sf_total_events" in leads_long.columns:
    leads_long["sf_engagement_score"] = minmax_score(leads_long["sf_total_events"])
else:
    leads_long["sf_engagement_score"] = 0.0

if "avg_inactive_gap_months" in leads_long.columns:
    leads_long["risk_penalty"] = minmax_score(leads_long["avg_inactive_gap_months"])
else:
    leads_long["risk_penalty"] = 0.0

# transparent weighted lead score
leads_long["lead_score"] = (
    0.45 * leads_long["model_score"]
    + 0.25 * leads_long["value_score"]
    + 0.20 * leads_long["activity_score"]
    + 0.10 * leads_long["sf_engagement_score"]
    - 0.10 * leads_long["risk_penalty"]
).clip(0, 1)

display(leads_long[[
    "ID_Customer",
    "product_group",
    "recommended_product",
    "model_probability",
    "model_score",
    "value_score",
    "activity_score",
    "sf_engagement_score",
    "risk_penalty",
    "lead_score"]].head(20))

,ID_Customer,product_group,recommended_product,model_probability,model_score,value_score,activity_score,sf_engagement_score,risk_penalty,lead_score
0,C00001,grp3,Premium Custom Type 2,0.1752,0.1752,0.0947,0.8343,0.7285,0.0179,0.3404
1,C00001,grp3,Premium Plus Type 2,0.1641,0.1641,0.0947,0.8343,0.7285,0.0179,0.3354
2,C00001,grp3,Repair Kits Type 2,0.1550,0.1550,0.0947,0.8343,0.7285,0.0179,0.3313
3,C00001,grp3,Sensors Type 2,0.1530,0.1530,0.0947,0.8343,0.7285,0.0179,0.3305
4,C00001,grp3,Monitoring Type 1,0.1192,0.1192,0.0947,0.8343,0.7285,0.0179,0.3153
5,C00001,grp3,Calibration Kits Type 1,0.1150,0.1150,0.0947,0.8343,0.7285,0.0179,0.3134
6,C00001,grp3,Controllers Type 1,0.1139,0.1139,0.0947,0.8343,0.7285,0.0179,0.3128
7,C00001,grp3,Core Basic Type 1,0.1113,0.1113,0.0947,0.8343,0.7285,0.0179,0.3117
8,C00001,grp3,Calibration Kits Type 2,0.1081,0.1081,0.0947,0.8343,0.7285,0.0179,0.3102
9,C00001,grp3,Controllers Type 2,0.0972,0.0972,0.0947,0.8343,0.7285,0.0179,0.3053


**Lead Tiers and Hot Reasons**

Lead tiers make the model easier to use for sales teams. We use quantile-based tiers:
- Hot: top 20%
- Warm: next 40%
- Cold: bottom 40%

The hot_reason column explains why a recommendation is attractive.
This is important because sales users need a reason, not just a score.

In [21]:
leads_long["lead_tier"] = pd.qcut(
    leads_long["lead_score"],
    q=[0.0, 0.40, 0.80, 1.0],
    labels=["Cold", "Warm", "Hot"],
    duplicates="drop")

def build_hot_reason(row: pd.Series) -> str:
    reasons = []

    if row.get("model_score", 0) >= 0.65:
        reasons.append("high model probability")
    if row.get("value_score", 0) >= 0.70:
        reasons.append("high customer value")
    if row.get("activity_score", 0) >= 0.65:
        reasons.append("recent/frequent buyer")
    if row.get("sf_engagement_score", 0) >= 0.65:
        reasons.append("strong Salesforce engagement")
    if row.get("risk_penalty", 0) <= 0.30:
        reasons.append("low inactivity risk")
    if row.get("sf_activity_without_recent_sales", 0) == 1:
        reasons.append("recent sales activity but no recent sales")

    return "; ".join(reasons) if reasons else "moderate model fit"


leads_long["hot_reason"] = leads_long.apply(build_hot_reason, axis=1)

display(leads_long["lead_tier"].value_counts(dropna=False).reset_index())
display(leads_long[[
    "ID_Customer",
    "cohort_ML",
    "product_group",
    "recommended_product",
    "lead_score",
    "lead_tier",
    "hot_reason",
]].sort_values("lead_score", ascending=False).head(20))

,lead_tier,count
0,Cold,24396
1,Warm,24396
2,Hot,12198


,ID_Customer,cohort_ML,product_group,recommended_product,lead_score,lead_tier,hot_reason
46435,C01524,Power,grp3,Training Type 2,0.6004,Hot,high customer value; recent/frequent buyer; st...
46436,C01524,Power,grp3,Extended Support Type 1,0.5942,Hot,high customer value; recent/frequent buyer; st...
46437,C01524,Power,grp3,Sensors Type 1,0.5925,Hot,high customer value; recent/frequent buyer; st...
46438,C01524,Power,grp3,Premium Standard Type 2,0.5859,Hot,high customer value; recent/frequent buyer; st...
46439,C01524,Power,grp3,Monitoring Type 2,0.5757,Hot,high customer value; recent/frequent buyer; st...
46440,C01524,Power,grp3,Controllers Type 2,0.5743,Hot,high customer value; recent/frequent buyer; st...
46441,C01524,Power,grp3,Installation Type 2,0.5685,Hot,high customer value; recent/frequent buyer; st...
46442,C01524,Power,grp3,Premium Plus Type 2,0.5675,Hot,high customer value; recent/frequent buyer; st...
46443,C01524,Power,grp3,Sensors Type 2,0.5664,Hot,high customer value; recent/frequent buyer; st...
4276,C00141,Power,grp3,Training Type 1,0.5628,Hot,high customer value; recent/frequent buyer; st...


**Select Final Top Recommendations per Customer**

A customer may have many possible recommendations.
For sales activation, we keep the top recommendations by lead_score, not only model probability.
This means a recommendation can rank higher when the customer is valuable, recently active, and commercially attractive.

In [22]:
lead_generation_top_recommendations = (
    leads_long
    .sort_values(
        ["ID_Customer", "lead_score", "model_probability"],
        ascending=[True, False, False])
    .copy())

lead_generation_top_recommendations["lead_rank_per_customer"] = (
    lead_generation_top_recommendations
    .groupby("ID_Customer")
    .cumcount()
    + 1)

lead_generation_top_recommendations = lead_generation_top_recommendations[
    lead_generation_top_recommendations["lead_rank_per_customer"] <= TOP_K
].copy()

display(lead_generation_top_recommendations.head(20))

,ID_Customer,product_group,recommended_product,model_probability,source_target_col,model_rank_per_customer,cohort_ML,cluster,Sales,historical_ltv,avg_monthly_revenue,active_months,months_since_last_purchase,activity_ratio_lifetime,avg_inactive_gap_months,prd_grp_1_coverage_pct,prd_grp_3_coverage_pct,sf_total_events,sf_active_months,sf_activity_without_recent_sales,sales_without_recent_sf_activity,model_score,value_score,recency_score,frequency_score,activity_score,sf_engagement_score,risk_penalty,lead_score,lead_tier,hot_reason,lead_rank_per_customer
0,C00001,grp3,Premium Custom Type 2,0.1752,purchase_flag_grp3_Premium Custom Type 2,1,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0,0.1752,0.0947,0.9714,0.6286,0.8343,0.7285,0.0179,0.3404,Hot,recent/frequent buyer; strong Salesforce engag...,1
1,C00001,grp3,Premium Plus Type 2,0.1641,purchase_flag_grp3_Premium Plus Type 2,2,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0,0.1641,0.0947,0.9714,0.6286,0.8343,0.7285,0.0179,0.3354,Hot,recent/frequent buyer; strong Salesforce engag...,2
2,C00001,grp3,Repair Kits Type 2,0.1550,purchase_flag_grp3_Repair Kits Type 2,3,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0,0.1550,0.0947,0.9714,0.6286,0.8343,0.7285,0.0179,0.3313,Hot,recent/frequent buyer; strong Salesforce engag...,3
32,C00002,grp3,Compliance Safety Type 1,0.1792,purchase_flag_grp3_Compliance Safety Type 1,1,Emerging,0,"4,738.6100","4,738.6100",152.8584,8,1,0.2581,3.2857,0.6667,0.2286,45,13,0,0,0.1792,0.0031,0.9714,0.2000,0.6629,0.2517,0.1173,0.2274,Warm,recent/frequent buyer; low inactivity risk,1
33,C00002,grp3,Core Advanced Type 1,0.1677,purchase_flag_grp3_Core Advanced Type 1,2,Emerging,0,"4,738.6100","4,738.6100",152.8584,8,1,0.2581,3.2857,0.6667,0.2286,45,13,0,0,0.1677,0.0031,0.9714,0.2000,0.6629,0.2517,0.1173,0.2223,Warm,recent/frequent buyer; low inactivity risk,2
34,C00002,grp3,Facility Safety Type 2,0.1640,purchase_flag_grp3_Facility Safety Type 2,3,Emerging,0,"4,738.6100","4,738.6100",152.8584,8,1,0.2581,3.2857,0.6667,0.2286,45,13,0,0,0.1640,0.0031,0.9714,0.2000,0.6629,0.2517,0.1173,0.2206,Warm,recent/frequent buyer; low inactivity risk,3
61,C00003,grp3,Core Basic Type 2,0.2303,purchase_flag_grp3_Core Basic Type 2,1,Core,4,"199,361.8000","199,361.8000","5,696.0514",31,1,0.8857,0.1333,0.8333,0.2857,124,24,0,0,0.2303,0.1307,0.9714,0.8571,0.9257,0.7748,0.0048,0.3985,Hot,recent/frequent buyer; strong Salesforce engag...,1
62,C00003,grp3,Personal Safety Type 1,0.2126,purchase_flag_grp3_Personal Safety Type 1,2,Core,4,"199,361.8000","199,361.8000","5,696.0514",31,1,0.8857,0.1333,0.8333,0.2857,124,24,0,0,0.2126,0.1307,0.9714,0.8571,0.9257,0.7748,0.0048,0.3905,Hot,recent/frequent buyer; strong Salesforce engag...,2
63,C00003,grp3,Premium Custom Type 2,0.1886,purchase_flag_grp3_Premium Custom Type 2,3,Core,4,"199,361.8000","199,361.8000","5,696.0514",31,1,0.8857,0.1333,0.8333,0.2857,124,24,0,0,0.1886,0.1307,0.9714,0.8571,0.9257,0.7748,0.0048,0.3797,Hot,recent/frequent buyer; strong Salesforce engag...,3
87,C00004,grp3,Core Basic Type 2,0.1898,purchase_flag_grp3_Core Basic Type 2,1,Core,4,"130,129.6200","130,129.6200","3,614.7117",26,0,0.7222,0.4000,0.5000,0.1143,100,23,0,0,0.1898,0.0853,1.0000,0.7143,0.8857,0.6159,0.0143,0.3440,Hot,recent/frequent buyer; low inactivity risk,1


**Create Final Customer-Level Output**

The final output contains one row per customer. It combines:
- customer context
- cohort label
- top model recommendations by product group
- final lead recommendations by lead score

In [ ]:
def make_lead_topk_wide(top_recs: pd.DataFrame, top_k: int = TOP_K) -> pd.DataFrame:
    wide_parts = []

    for k in range(1, top_k + 1):
        kth = (
            top_recs[top_recs["lead_rank_per_customer"] == k]
            [[
                "ID_Customer",
                "product_group",
                "recommended_product",
                "model_probability",
                "lead_score",
                "lead_tier",
                "hot_reason"]]
            .rename(columns={
                "product_group": f"lead_top{k}_product_group",
                "recommended_product": f"lead_top{k}_product",
                "model_probability": f"lead_top{k}_model_probability",
                "lead_score": f"lead_top{k}_score",
                "lead_tier": f"lead_top{k}_tier",
                "hot_reason": f"lead_top{k}_reason"}))
        wide_parts.append(kth)

    final_wide = wide_parts[0]
    for kth in wide_parts[1:]:
        final_wide = final_wide.merge(kth, on="ID_Customer", how="outer")

    return final_wide

lead_recos_wide = make_lead_topk_wide(lead_generation_top_recommendations, TOP_K)

customer_context_final_cols = [
    c for c in [
        "ID_Customer",
        "cohort_ML",
        "cluster",
        "Sales",
        "historical_ltv",
        "avg_monthly_revenue",
        "active_months",
        "months_since_last_purchase",
        "activity_ratio_lifetime",
        "sf_total_events",
        "sf_active_months",
        "sf_months_since_last_event"]
    if c in df_customers.columns]

lead_generation_final = (
    df_customers[customer_context_final_cols]
    .merge(recos_grp1_top3_wide, on="ID_Customer", how="left")
    .merge(recos_grp3_top3_wide, on="ID_Customer", how="left")
    .merge(lead_recos_wide, on="ID_Customer", how="left"))

display(lead_generation_final.head())
print("Final customer-level output:", lead_generation_final.shape)

,ID_Customer,cohort_ML,cluster,Sales,historical_ltv,avg_monthly_revenue,active_months,months_since_last_purchase,activity_ratio_lifetime,sf_total_events,sf_active_months,reco_top1_grp1_product,reco_top1_grp1_probability,reco_top1_grp1_rank,reco_top2_grp1_product,reco_top2_grp1_probability,reco_top2_grp1_rank,reco_top3_grp1_product,reco_top3_grp1_probability,reco_top3_grp1_rank,reco_top1_grp3_product,reco_top1_grp3_probability,reco_top1_grp3_rank,reco_top2_grp3_product,reco_top2_grp3_probability,reco_top2_grp3_rank,reco_top3_grp3_product,reco_top3_grp3_probability,reco_top3_grp3_rank,lead_top1_product_group,lead_top1_product,lead_top1_model_probability,lead_top1_score,lead_top1_tier,lead_top1_reason,lead_top2_product_group,lead_top2_product,lead_top2_model_probability,lead_top2_score,lead_top2_tier,lead_top2_reason,lead_top3_product_group,lead_top3_product,lead_top3_model_probability,lead_top3_score,lead_top3_tier,lead_top3_reason
0,C00001,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,117,29,Service Contracts,0.0747,1.0000,Safety Solutions,0.0644,2.0000,NaN,NaN,NaN,Premium Custom Type 2,0.1752,1,Premium Plus Type 2,0.1641,2,Repair Kits Type 2,0.1550,3,grp3,Premium Custom Type 2,0.1752,0.3404,Hot,recent/frequent buyer; strong Salesforce engag...,grp3,Premium Plus Type 2,0.1641,0.3354,Hot,recent/frequent buyer; strong Salesforce engag...,grp3,Repair Kits Type 2,0.1550,0.3313,Hot,recent/frequent buyer; strong Salesforce engag...
1,C00002,Emerging,0,"4,738.6100","4,738.6100",152.8584,8,1,0.2581,45,13,Maintenance Kits,0.0861,1.0000,Service Contracts,0.0836,2.0000,NaN,NaN,NaN,Compliance Safety Type 1,0.1792,1,Core Advanced Type 1,0.1677,2,Facility Safety Type 2,0.1640,3,grp3,Compliance Safety Type 1,0.1792,0.2274,Warm,recent/frequent buyer; low inactivity risk,grp3,Core Advanced Type 1,0.1677,0.2223,Warm,recent/frequent buyer; low inactivity risk,grp3,Facility Safety Type 2,0.1640,0.2206,Warm,recent/frequent buyer; low inactivity risk
2,C00003,Core,4,"199,361.8000","199,361.8000","5,696.0514",31,1,0.8857,124,24,Automation Modules,0.1262,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,Core Basic Type 2,0.2303,1,Personal Safety Type 1,0.2126,2,Premium Custom Type 2,0.1886,3,grp3,Core Basic Type 2,0.2303,0.3985,Hot,recent/frequent buyer; strong Salesforce engag...,grp3,Personal Safety Type 1,0.2126,0.3905,Hot,recent/frequent buyer; strong Salesforce engag...,grp3,Premium Custom Type 2,0.1886,0.3797,Hot,recent/frequent buyer; strong Salesforce engag...
3,C00004,Core,4,"130,129.6200","130,129.6200","3,614.7117",26,0,0.7222,100,23,Premium Products,0.0707,1.0000,Safety Solutions,0.0702,2.0000,Service Contracts,0.0353,3.0000,Core Basic Type 2,0.1898,1,Controllers Type 1,0.1674,2,Monitoring Type 1,0.1459,3,grp3,Core Basic Type 2,0.1898,0.3440,Hot,recent/frequent buyer; low inactivity risk,grp3,Controllers Type 1,0.1674,0.3340,Hot,recent/frequent buyer; low inactivity risk,grp3,Monitoring Type 1,0.1459,0.3243,Hot,recent/frequent buyer; low inactivity risk
4,C00005,Emerging,0,"39,122.4500","39,122.4500","1,304.0817",17,5,0.5667,82,23,Safety Solutions,0.0795,1.0000,Maintenance Kits,0.0511,2.0000,NaN,NaN,NaN,Training Type 1,0.1787,1,Extended Support Type 1,0.1652,2,Training Type 2,0.1428,3,grp3,Training Type 1,0.1787,0.2730,Hot,recent/frequent buyer; low inactivity risk,grp3,Extended Support Type 1,0.1652,0.2669,Hot,recent/frequent buyer; low inactivity risk,grp3,Training Type 2,0.1428,0.2569,Warm,recent/frequent buyer; low inactivity risk


Final customer-level output: (1993, 47)


**Final Business Summary**

Useful views:
- lead tier distribution
- hot leads by cohort
- top recommended products
- average lead score by cohort

In [24]:
lead_tier_summary = (
    leads_long
    .groupby("lead_tier", observed=False)
    .agg(
        recommendation_rows=("ID_Customer", "count"),
        customers=("ID_Customer", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        avg_model_probability=("model_probability", "mean"))
    .reset_index())

cohort_lead_summary = (
    lead_generation_top_recommendations
    .groupby(["cohort_ML", "lead_tier"], observed=False)
    .agg(
        recommendation_rows=("ID_Customer", "count"),
        customers=("ID_Customer", "nunique"),
        avg_lead_score=("lead_score", "mean"))
    .reset_index()
    .sort_values(["cohort_ML", "avg_lead_score"], ascending=[True, False]))

product_lead_summary = (
    lead_generation_top_recommendations
    .groupby(["product_group", "recommended_product"], as_index=False)
    .agg(
        recommendation_rows=("ID_Customer", "count"),
        customers=("ID_Customer", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        avg_model_probability=("model_probability", "mean"))
    .sort_values("recommendation_rows", ascending=False))

display(lead_tier_summary)
display(cohort_lead_summary.head(30))
display(product_lead_summary.head(30))

,lead_tier,recommendation_rows,customers,avg_lead_score,avg_model_probability
0,Cold,24396,1208,0.1284,0.0473
1,Warm,24396,1513,0.2118,0.0870
2,Hot,12198,736,0.3219,0.1014


,cohort_ML,lead_tier,recommendation_rows,customers,avg_lead_score
2,Core,Hot,962,322,0.3541
1,Core,Warm,4,2,0.2581
0,Core,Cold,0,0,NaN
4,Dormant,Warm,80,35,0.1944
3,Dormant,Cold,403,141,0.1366
5,Dormant,Hot,0,0,NaN
8,Emerging,Hot,842,353,0.2909
7,Emerging,Warm,2957,1078,0.2222
6,Emerging,Cold,428,174,0.1476
11,Occasional,Hot,4,2,0.2845


,product_group,recommended_product,recommendation_rows,customers,avg_lead_score,avg_model_probability
15,grp3,Core Basic Type 2,342,342,0.2464,0.1766
8,grp3,Compliance Safety Type 1,328,328,0.2545,0.1891
22,grp3,Monitoring Type 1,320,320,0.2409,0.1677
14,grp3,Core Basic Type 1,310,310,0.2434,0.1740
24,grp3,Personal Safety Type 1,305,305,0.2616,0.1900
12,grp3,Core Advanced Type 1,303,303,0.2432,0.1777
36,grp3,Sensors Type 1,291,291,0.2604,0.1751
10,grp3,Controllers Type 1,279,279,0.2435,0.1698
27,grp3,Premium Custom Type 2,271,271,0.2615,0.1795
20,grp3,Installation Type 1,267,267,0.2356,0.1845


In [25]:
# save Outputs
outputs = {
    "recos_grp1_top3_wide": recos_grp1_top3_wide,
    "recos_grp3_top3_wide": recos_grp3_top3_wide,
    "recommendations_long": recommendations_long,
    "recommendations_topk_model": recommendations_topk_model,
    "model_registry_all_products": model_registry_all,
    "leads_long_scored": leads_long,
    "lead_generation_top_recommendations": lead_generation_top_recommendations,
    "lead_generation_final": lead_generation_final,
    "lead_tier_summary": lead_tier_summary,
    "cohort_lead_summary": cohort_lead_summary,
    "product_lead_summary": product_lead_summary}

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)

Saved files:
- cohort_lead_summary.csv
- lead_generation_final.csv
- lead_generation_top_recommendations.csv
- lead_tier_summary.csv
- leads_long_scored.csv
- model_registry_all_products.csv
- product_lead_summary.csv
- recommendations_long.csv
- recommendations_topk_model.csv
- recos_grp1_top3_wide.csv
- recos_grp3_top3_wide.csv
